Imports

In [130]:
import csv
import json
from datetime import datetime

Constantes

In [131]:
CAMINHO_ARQUIVO = "transacoes.csv"
LIMITE_SUSPEITO = 10000.00

Lê o CSV e retorna a lista de transações brutas

In [132]:
def ler_transacoes(caminho):
    try:
        with open(caminho, mode="r", encoding="utf-8", newline="") as arquivo:
            leitor = csv.DictReader(arquivo)
            registros = list(leitor)
    except FileNotFoundError:
        print(f"Erro: o arquivo '{caminho}' não foi encontrado.")
        return []

    print(f"{len(registros)} registros lidos de '{caminho}'.")
    return registros


transacoes = ler_transacoes(CAMINHO_ARQUIVO)

# Acesso pelo nome da coluna:
for t in transacoes[:3]:
    print(t["id"], t["data"], t["cliente_id"], t["valor"])


202 registros lidos de 'transacoes.csv'.
1 2026-01-01 CLI011 108.52
2 2026-01-01 CLI010 103.77
3 2026-01-02 CLI013 192.05


Valida uma única linha e retorna o registro limpo

In [143]:
def validar_transacao(linha):
    # Valida uma linha. Retorna o registro limpo ou None se a linha for inválida.
    id_txt = (linha["id"] or "").strip()
    cliente = (linha["cliente_id"] or "").strip()
    data = (linha["data"] or "").strip()
    tipo = (linha["tipo"] or "").strip()
    valor_txt = (linha["valor"] or "").strip()


    # id vazio ou não numérico
    if not id_txt.isdigit():
        return None

    # cliente_id vazio
    if cliente == "" or cliente.lower() in ["null", "none", "nan", "n/a"]:
        return None

    # data diferente de AAAA-MM-DD
    if len(data) != 10:
        return None
    try:
        data_obj = datetime.strptime(data, "%Y-%m-%d")
    except ValueError:
        return None

    # tipo diferente de credito ou debito
    if tipo not in ["credito", "debito"]:
        return None

    # valor não numérico ou <= 0
    if valor_txt.lower() == "nan":
        return None
    try:
        valor = float(valor_txt)
    except ValueError:
        return None
    if valor <= 0:
        return None

    # devolve o registro limpo
    return {
        "id": int(id_txt),
        "data": data_obj,
        "cliente_id": cliente,
        "tipo": tipo,
        "valor": valor,
        "descricao": (linha["descricao"] or "").strip(),
        "categoria": (linha["categoria"] or "").strip(),
    }

print(validar_transacao({"id": "1", "data": "2026-01-05", "cliente_id": "CLI001", "tipo": "credito",
                         "valor": "3500.00", "descricao": "Salário janeiro", "categoria": "salario"}))
print(validar_transacao({"id": "x99", "data": "2026-01-05", "cliente_id": "CLI001", "tipo": "credito",
                         "valor": "3500.00", "descricao": "Salário janeiro", "categoria": "salario"}))

{'id': 1, 'data': datetime.datetime(2026, 1, 5, 0, 0), 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': 3500.0, 'descricao': 'Salário janeiro', 'categoria': 'salario'}
None


In [134]:
def limpar_transacoes(transacoes):
    transacoes_trat = []
    ids_vistos = []

    for linha in transacoes:
        registro = validar_transacao(linha)
        if registro is None:              # linha inválida: descarta
            continue
        if registro["id"] in ids_vistos:  # duplicada: descarta
            continue

        ids_vistos.append(registro["id"])
        transacoes_trat.append(registro)

    total = len(transacoes)
    validas = len(transacoes_trat)
    print(f"Total de linhas lidas: {total}")
    print(f"Linhas válidas: {validas}")
    print(f"Linhas inválidas: {total - validas}")

    return transacoes_trat
transacoes_trat = limpar_transacoes(transacoes)

Total de linhas lidas: 202
Linhas válidas: 148
Linhas inválidas: 54


Quantos dias se passaram desde a transação mais antiga e a mais recente presentes nos dados?

In [135]:
# Extrair o mês AAAA-MM
for transacao in transacoes_trat:
    transacao["mes"] = transacao["data"].strftime("%Y-%m")

# Dias entre a transação mais antiga e a mais recente
datas = []
for transacao in transacoes_trat:
    datas.append(transacao["data"])

data_mais_antiga = min(datas)
data_mais_recente = max(datas)
diferenca = data_mais_recente - data_mais_antiga

print(f"Transação mais antiga: {data_mais_antiga.strftime('%d/%m/%Y')}")
print(f"Transação mais recente: {data_mais_recente.strftime('%d/%m/%Y')}")
print(f"Dias entre elas: {diferenca.days}")

Transação mais antiga: 01/01/2026
Transação mais recente: 28/06/2026
Dias entre elas: 178


Agrupar transações por mês

In [136]:
def agrupar_por_mes(transacoes_trat):
    # Retorna um dicionário {mes: [lista de transações daquele mês]}.
    grupos = {}

    for transacao in transacoes_trat:
        mes = transacao["data"].strftime("%Y-%m")
        if mes not in grupos:
            grupos[mes] = []          # cria a lista vazia na 1ª vez
        grupos[mes].append(transacao)

    return grupos

transacoes_por_mes = agrupar_por_mes(transacoes_trat)
print(transacoes_por_mes.keys())

dict_keys(['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06'])


Métricas mensais

In [144]:
def calcular_metricas_mensais():
    # Agrupa transacoes_trat por mês e retorna as métricas de todos os meses.
    transacoes_por_mes = agrupar_por_mes(transacoes_trat)   # usa a variável do notebook
    metricas = {}

    for mes in sorted(transacoes_por_mes):
        lista = transacoes_por_mes[mes]
        total_creditos = 0
        total_debitos = 0
        maior = lista[0]
        menor = lista[0]

        for t in lista:
            if t["tipo"] == "credito":
                total_creditos += t["valor"]
            else:
                total_debitos += t["valor"]
            if t["valor"] > maior["valor"]:
                maior = t
            if t["valor"] < menor["valor"]:
                menor = t

        quantidade = len(lista)

        metricas[mes] = {
            "quantidade": quantidade,
            "total_credito": round(total_creditos, 2),
            "total_debito": round(total_debitos, 2),
            "saldo": round(total_creditos - total_debitos, 2),
            "media": round((total_creditos + total_debitos) / quantidade, 2),
            "maior_valor": maior["valor"],
            "menor_valor": menor["valor"],
        }

    return metricas


metricas_mensais = calcular_metricas_mensais()
print(metricas_mensais["2026-01"])

{'quantidade': 29, 'total_credito': 25194.19, 'total_debito': 16932.69, 'saldo': 8261.5, 'media': 1452.65, 'maior_valor': 12500.0, 'menor_valor': 22.33}


Verificar transações suspeitas

In [138]:
def moeda(valor):
    # Converte 3500.0 em 'R$ 3.500,00'.
    texto = f"{valor:,.2f}"
    texto = texto.replace(",", "X").replace(".", ",").replace("X", ".")
    return f"R$ {texto}"

def identificar_suspeitas(transacoes):
    # Retorna as transações com valor acima de LIMITE_SUSPEITO (lista vazia se não houver).
    suspeitas = []
    for t in transacoes:
        if t["valor"] > LIMITE_SUSPEITO:
            suspeitas.append(t)
    return suspeitas

def exibir_suspeitas(suspeitas):
    # Imprime as transações suspeitas.
    print()
    print("===== TRANSAÇÕES SUSPEITAS =====")
    if len(suspeitas) == 0:
        print("Nenhuma transação suspeita encontrada.")
        return
    for t in suspeitas:
        data_txt = t["data"].strftime("%Y-%m-%d")
        print(f"ID: {t['id']} | Cliente: {t['cliente_id']} | Data: {data_txt} | Valor: {moeda(t['valor'])}")

transacoes_suspeitas = identificar_suspeitas(transacoes_trat)
exibir_suspeitas(transacoes_suspeitas)


===== TRANSAÇÕES SUSPEITAS =====
ID: 25 | Cliente: CLI010 | Data: 2026-01-19 | Valor: R$ 12.500,00
ID: 124 | Cliente: CLI004 | Data: 2026-05-03 | Valor: R$ 10.500,00
ID: 139 | Cliente: CLI006 | Data: 2026-05-14 | Valor: R$ 31.000,00
ID: 140 | Cliente: CLI011 | Data: 2026-05-16 | Valor: R$ 22.800,00
ID: 185 | Cliente: CLI010 | Data: 2026-06-15 | Valor: R$ 18.350,75
ID: 190 | Cliente: CLI008 | Data: 2026-06-25 | Valor: R$ 15.000,00


Exibir relatórios mensais

In [139]:


def exibir_relatorio(mes=None):
    # Mostra o relatório de um mês específico, ou de todos se nenhum mês for passado.

    if mes is None:
        meses = list(metricas_mensais.keys())   # todos os meses
        transacoes_periodo = transacoes_trat    # período geral
    elif mes in metricas_mensais:
        meses = [mes]                                  # só o mês pedido
        transacoes_periodo = transacoes_por_mes[mes]   # período só desse mês
    else:
        print(f"\nMês {mes} não encontrado nos dados.")
        return
    datas = []
    for t in transacoes_periodo:
        datas.append(t["data"])
    inicio = min(datas)
    fim = max(datas)
    print()
    print(f"Período analisado: {inicio.strftime('%Y-%m-%d')} → {fim.strftime('%Y-%m-%d')} ({(fim - inicio).days} dias)")
    for m_atual in meses:
        m = metricas_mensais[m_atual]
        print()
        print("===== RELATÓRIO MENSAL =====")
        print(f"Mês: {m_atual}")
        print(f"  Transações: {m['quantidade']}")
        print(f"  Total crédito: {moeda(m['total_credito'])}")
        print(f"  Total débito:  {moeda(m['total_debito'])}")
        print(f"  Saldo:         {moeda(m['saldo'])}")
        print(f"  Média:         {moeda(m['media'])}")
        print(f"  Maior valor:   {moeda(m['maior_valor'])}")
        print(f"  Menor valor:   {moeda(m['menor_valor'])}")

        # suspeitas só deste mês
        exibir_suspeitas(identificar_suspeitas(transacoes_por_mes[m_atual]))


exibir_relatorio()


Período analisado: 2026-01-01 → 2026-06-28 (178 dias)

===== RELATÓRIO MENSAL =====
Mês: 2026-01
  Transações: 29
  Total crédito: R$ 25.194,19
  Total débito:  R$ 16.932,69
  Saldo:         R$ 8.261,50
  Média:         R$ 1.452,65
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 22,33

===== TRANSAÇÕES SUSPEITAS =====
ID: 25 | Cliente: CLI010 | Data: 2026-01-19 | Valor: R$ 12.500,00

===== RELATÓRIO MENSAL =====
Mês: 2026-02
  Transações: 17
  Total crédito: R$ 30.251,99
  Total débito:  R$ 4.885,42
  Saldo:         R$ 25.366,57
  Média:         R$ 2.066,91
  Maior valor:   R$ 7.200,00
  Menor valor:   R$ 39,38

===== TRANSAÇÕES SUSPEITAS =====
Nenhuma transação suspeita encontrada.

===== RELATÓRIO MENSAL =====
Mês: 2026-03
  Transações: 21
  Total crédito: R$ 30.233,11
  Total débito:  R$ 1.380,44
  Saldo:         R$ 28.852,67
  Média:         R$ 1.505,41
  Maior valor:   R$ 7.200,00
  Menor valor:   R$ 33,63

===== TRANSAÇÕES SUSPEITAS =====
Nenhuma transação suspeita encontrada.

Exportar JSON

In [140]:
def salvar_relatorio_json():
    # Monta o relatório com os resultados do notebook e salva em JSON.
    caminho="relatorio.json"
    suspeitas_json = []
    for t in transacoes_suspeitas:
        suspeitas_json.append({
            "id": t["id"],
            "cliente_id": t["cliente_id"],
            "data": t["data"].strftime("%Y-%m-%d"),
            "valor": t["valor"],
        })

    relatorio = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": len(transacoes_trat),
        "total_transacoes_invalidas": len(transacoes) - len(transacoes_trat),
        "resumo_mensal": metricas_mensais,
        "transacoes_suspeitas": suspeitas_json,
    }

    with open(caminho, "w", encoding="utf-8") as arquivo:
        json.dump(relatorio, arquivo, indent=2, ensure_ascii=False)

    print(f"Relatório salvo em '{caminho}'.")
    return relatorio

relatorio = salvar_relatorio_json()



Relatório salvo em 'relatorio.json'.


Execução principal

In [141]:
# ===== CÉLULA DE EXECUÇÃO PRINCIPAL =====

transacoes = ler_transacoes(CAMINHO_ARQUIVO)

if len(transacoes) == 0:
    print("Nenhum dado para processar.")
else:
    transacoes_trat = limpar_transacoes(transacoes)
    transacoes_por_mes = agrupar_por_mes(transacoes_trat)
    metricas_mensais = calcular_metricas_mensais()
    transacoes_suspeitas = identificar_suspeitas(transacoes_trat)

    exibir_relatorio()
    exibir_suspeitas(transacoes_suspeitas)
    salvar_relatorio_json()

202 registros lidos de 'transacoes.csv'.
Total de linhas lidas: 202
Linhas válidas: 148
Linhas inválidas: 54

Período analisado: 2026-01-01 → 2026-06-28 (178 dias)

===== RELATÓRIO MENSAL =====
Mês: 2026-01
  Transações: 29
  Total crédito: R$ 25.194,19
  Total débito:  R$ 16.932,69
  Saldo:         R$ 8.261,50
  Média:         R$ 1.452,65
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 22,33

===== TRANSAÇÕES SUSPEITAS =====
ID: 25 | Cliente: CLI010 | Data: 2026-01-19 | Valor: R$ 12.500,00

===== RELATÓRIO MENSAL =====
Mês: 2026-02
  Transações: 17
  Total crédito: R$ 30.251,99
  Total débito:  R$ 4.885,42
  Saldo:         R$ 25.366,57
  Média:         R$ 2.066,91
  Maior valor:   R$ 7.200,00
  Menor valor:   R$ 39,38

===== TRANSAÇÕES SUSPEITAS =====
Nenhuma transação suspeita encontrada.

===== RELATÓRIO MENSAL =====
Mês: 2026-03
  Transações: 21
  Total crédito: R$ 30.233,11
  Total débito:  R$ 1.380,44
  Saldo:         R$ 28.852,67
  Média:         R$ 1.505,41
  Maior valor:   R